# 🛒 SmartPantry AI — Sprint 2: Analytics Engine
**Portfolio Project | Agentic AI + Advanced Analytics**

### Prerequisite
Sprint 1 must have been run first in this Colab session so `data/` CSVs exist.
If you're in a fresh session: run Sprint 1 Cell 1 (clone repo) + Cells 2–7 to regenerate data, then come back here.

### Four modules
| Module | FMCG equivalent |
|---|---|
| Depletion Forecast | Shelf velocity + promo uplift → stockout risk |
| Order Optimisation | Replenishment planning with budget caps |
| Menu Coverage Check | Demand-supply matching against a campaign |
| Spend Analytics | Category P&L vs trade investment |

## Cell 1 — Clone Repo (run every session)

In [ ]:
import os, subprocess

GITHUB_USER = 'your-github-username'   # ← UPDATE THIS
GITHUB_REPO = 'SmartPantry-AI'

REPO_DIR   = f'/content/{GITHUB_REPO}'
DATA_DIR   = f'{REPO_DIR}/data'
OUTPUT_DIR = f'{REPO_DIR}/outputs'

if os.path.exists(REPO_DIR):
    subprocess.run('git pull', shell=True, cwd=REPO_DIR)
    print('Repo pulled ✓')
else:
    subprocess.run(f'git clone https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git {REPO_DIR}', shell=True)
    print('Repo cloned ✓')

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data files: {os.listdir(DATA_DIR)}')

## Cell 2 — Imports & Load Sprint 1 Data

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta

np.random.seed(42)
today = datetime.today().date()

df_master  = pd.read_csv(f'{DATA_DIR}/item_master.csv')
df_pantry  = pd.read_csv(f'{DATA_DIR}/pantry_snapshot.csv')
df_history = pd.read_csv(f'{DATA_DIR}/consumption_history.csv')
df_demand  = pd.read_csv(f'{DATA_DIR}/menu_demand.csv')
with open(f'{DATA_DIR}/family_profile.json') as f:
    profile = json.load(f)

print('Sprint 1 data loaded ✓')
print(f'  Items: {len(df_master)} | History rows: {len(df_history)} | Menu ingredients: {len(df_demand)}')
print(f'  Family: {profile["total_members"]} members | Budget: ₹{profile["monthly_grocery_budget_inr"]:,}/mo')

## Cell 3 — Module 1: Depletion Forecast
**FMCG mapping:** velocity + promo uplift → days of supply + replenishment priority index

In [ ]:
def compute_depletion_forecast(df_pantry, df_history, df_demand):
    df_hist = df_history.copy()
    df_hist['week_start'] = pd.to_datetime(df_hist['week_start'])

    def weighted_avg(grp):
        grp = grp.sort_values('week_start')
        n = len(grp)
        weights = np.array([1]*max(0,n-4) + [2]*min(4,n), dtype=float)[-n:]
        return np.average(grp['consumed_qty'].values, weights=weights)

    hist_rates = (
        df_hist.groupby('item_name')
        .apply(weighted_avg, include_groups=False)
        .reset_index(name='weighted_weekly_rate')
    )
    cv_df = (
        df_hist.groupby('item_name')['consumed_qty']
        .agg(['mean','std'])
        .assign(cv=lambda x: (x['std']/x['mean']*100).round(1))
        .reset_index()[['item_name','cv']]
    )

    df_fc = df_pantry[['item_id','item_name','category','unit','current_stock',
                        'daily_consumption_rate','reorder_threshold','lead_time_days',
                        'price_inr_per_unit','stock_status','reorder_needed',
                        'is_perishable','shelf_life_days']].copy()
    df_fc = df_fc.merge(hist_rates, on='item_name', how='left')
    df_fc = df_fc.merge(cv_df,      on='item_name', how='left')
    df_fc = df_fc.merge(df_demand,  on='item_name', how='left')

    df_fc['weighted_weekly_rate'] = df_fc['weighted_weekly_rate'].fillna(df_fc['daily_consumption_rate']*7)
    df_fc['weekly_menu_demand']   = df_fc['weekly_menu_demand'].fillna(0)
    df_fc['cv']                   = df_fc['cv'].fillna(10)

    df_fc['hist_daily_rate']       = (df_fc['weighted_weekly_rate']/7).round(4)
    df_fc['menu_daily_demand']     = (df_fc['weekly_menu_demand']/7).round(4)
    df_fc['combined_daily_demand'] = df_fc[['hist_daily_rate','menu_daily_demand']].max(axis=1).round(4)
    df_fc['menu_uplift_pct']       = np.where(
        df_fc['hist_daily_rate']>0,
        ((df_fc['menu_daily_demand']-df_fc['hist_daily_rate'])/df_fc['hist_daily_rate']*100).round(1), 0
    )
    df_fc['forecast_days_to_empty'] = np.where(
        df_fc['combined_daily_demand']>0,
        (df_fc['current_stock']/df_fc['combined_daily_demand']).clip(0,60).round(1), 60.0
    )
    df_fc['urgency_score'] = (
        (1 - df_fc['forecast_days_to_empty'].clip(0,14)/14)*60 +
        (df_fc['cv'].clip(0,30)/30)*20 +
        (df_fc['menu_uplift_pct'].clip(0,100)/100)*20
    ).round(1).clip(0,100)
    df_fc['must_order_by'] = df_fc.apply(
        lambda r: str(today + timedelta(days=max(0, r['forecast_days_to_empty']-r['lead_time_days']-1))), axis=1
    )
    df_fc['forecast_status'] = pd.cut(
        df_fc['forecast_days_to_empty'], bins=[-1,3,7,14,60],
        labels=['CRITICAL','LOW','MODERATE','OK']
    )
    return df_fc.sort_values('urgency_score', ascending=False).reset_index(drop=True)

df_forecast = compute_depletion_forecast(df_pantry, df_history, df_demand)
df_forecast.to_csv(f'{DATA_DIR}/forecast_output.csv', index=False)

sc = df_forecast['forecast_status'].value_counts()
print('Depletion Forecast ✓')
for s,e in [('CRITICAL','🔴'),('LOW','🟠'),('MODERATE','🟡'),('OK','🟢')]:
    print(f'  {e} {s}: {sc.get(s,0)} items')
print(f'  Menu uplift >20%: {(df_forecast["menu_uplift_pct"]>20).sum()} items')
print()
cols = ['item_name','current_stock','unit','forecast_days_to_empty','menu_uplift_pct','urgency_score','must_order_by']
print('Top 8 by urgency:')
print(df_forecast[cols].head(8).to_string(index=False))
print(f'\nSaved → {DATA_DIR}/forecast_output.csv')

## Cell 4 — Module 2: Order Optimisation

In [ ]:
def build_order_list(df_forecast, monthly_budget_inr, horizon_days=7):
    weekly_budget = monthly_budget_inr / 4
    df = df_forecast.copy()
    df['qty_to_order'] = np.maximum(0, (df['combined_daily_demand']*14) - df['current_stock']).round(2)
    df['order_flag']   = 'SKIP'
    df.loc[df['reorder_needed'] | (df['forecast_days_to_empty']<=horizon_days), 'order_flag'] = 'MUST ORDER'
    df.loc[(df['order_flag']=='SKIP') & (df['urgency_score']>50) & (df['forecast_days_to_empty']<=14), 'order_flag'] = 'RECOMMENDED'
    df['order_cost_inr'] = (df['qty_to_order'] * df['price_inr_per_unit']).round(0)

    candidates = df[df['order_flag'].isin(['MUST ORDER','RECOMMENDED'])].copy()
    candidates = candidates.sort_values(['order_flag','urgency_score'], ascending=[True,False])
    cumulative, selected = 0, []
    for _, row in candidates.iterrows():
        cost = row['order_cost_inr']
        if row['order_flag']=='MUST ORDER':
            selected.append(True); cumulative += cost
        elif cumulative + cost <= weekly_budget:
            selected.append(True); cumulative += cost
        else:
            selected.append(False)
    candidates['within_budget'] = selected
    return candidates[candidates['within_budget']], cumulative, weekly_budget

df_order, total_cost, weekly_budget = build_order_list(df_forecast, profile['monthly_grocery_budget_inr'])
df_order.to_csv(f'{DATA_DIR}/order_list.csv', index=False)

must = (df_order['order_flag']=='MUST ORDER').sum()
rec  = (df_order['order_flag']=='RECOMMENDED').sum()
print(f'Order List ✓  |  Weekly budget: ₹{weekly_budget:,.0f}')
print(f'  Must order  : {must} items')
print(f'  Recommended : {rec} items')
print(f'  Total cost  : ₹{total_cost:,.0f}  ({total_cost/weekly_budget*100:.0f}% of budget)')
print(f'  Remaining   : ₹{weekly_budget-total_cost:,.0f}')
print()
print(df_order[['item_name','unit','qty_to_order','order_cost_inr','order_flag','urgency_score']].to_string(index=False))
print(f'\nSaved → {DATA_DIR}/order_list.csv')

## Cell 5 — Module 3: Menu Coverage Check

In [ ]:
def check_menu_coverage(df_forecast, df_demand):
    df = df_demand.merge(
        df_forecast[['item_name','current_stock','unit','forecast_days_to_empty','urgency_score']],
        on='item_name', how='left'
    )
    df['stock_covers_menu'] = df['current_stock'] >= df['weekly_menu_demand']
    df['shortfall']         = (df['weekly_menu_demand'] - df['current_stock']).clip(lower=0).round(3)
    df['coverage_pct']      = (df['current_stock']/df['weekly_menu_demand']*100).clip(0,100).round(1)
    df['risk_level']        = pd.cut(df['coverage_pct'], bins=[-1,50,80,100,101],
                                     labels=['HIGH RISK','MEDIUM RISK','LOW RISK','COVERED'])
    return df.sort_values('coverage_pct').reset_index(drop=True)

df_coverage = check_menu_coverage(df_forecast, df_demand)
df_coverage.to_csv(f'{DATA_DIR}/menu_coverage.csv', index=False)

covered   = df_coverage['stock_covers_menu'].sum()
uncovered = (~df_coverage['stock_covers_menu']).sum()
print(f'Menu Coverage ✓  |  {covered}/{len(df_coverage)} ingredients covered')
if uncovered > 0:
    print(f'  ⚠️  {uncovered} gaps — meals at risk:')
    print(df_coverage[~df_coverage['stock_covers_menu']][
        ['item_name','unit','weekly_menu_demand','current_stock','shortfall','coverage_pct']
    ].to_string(index=False))
else:
    print('  ✅ All ingredients covered!')
print(f'\nSaved → {DATA_DIR}/menu_coverage.csv')

## Cell 6 — Module 4: Spend Analytics

In [ ]:
def compute_spend_analytics(df_fc, profile):
    df_sp = df_fc.copy()
    df_sp['monthly_spend_inr'] = (df_sp['combined_daily_demand']*30*df_sp['price_inr_per_unit']).round(0)
    by_cat = df_sp.groupby('category')['monthly_spend_inr'].sum().sort_values(ascending=False)
    total  = by_cat.sum()
    budget = profile['monthly_grocery_budget_inr']
    top5   = df_sp.nlargest(5,'monthly_spend_inr')[['item_name','category','monthly_spend_inr']]
    return {
        'total_monthly_est': round(total,0),
        'budget': budget,
        'utilisation_pct': round(total/budget*100,1),
        'budget_headroom': round(budget-total,0),
        'by_category': by_cat.round(0).to_dict(),
        'by_category_pct': (by_cat/total*100).round(1).to_dict(),
        'top5_items': top5.to_dict('records')
    }

spend = compute_spend_analytics(df_forecast, profile)
with open(f'{DATA_DIR}/spend_analytics.json','w') as f:
    json.dump(spend, f, indent=2)

print(f'Spend Analytics ✓')
print(f'  Monthly est  : ₹{spend["total_monthly_est"]:,.0f}')
print(f'  Budget       : ₹{spend["budget"]:,}')
print(f'  Utilisation  : {spend["utilisation_pct"]}%')
print(f'  Headroom     : ₹{spend["budget_headroom"]:,.0f}')
print()
for cat, amt in spend['by_category'].items():
    pct = spend['by_category_pct'][cat]
    print(f'  {cat:<18} ₹{amt:>6,.0f}  ({pct:.0f}%)')
print(f'\nSaved → {DATA_DIR}/spend_analytics.json')

## Cell 7 — Dashboard (4 charts)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16,12))
fig.suptitle('SmartPantry AI — Sprint 2: Analytics Dashboard', fontsize=15, fontweight='bold')
SC = {'CRITICAL':'#E53935','LOW':'#FB8C00','MODERATE':'#FDD835','OK':'#43A047'}

# Chart 1: Urgency scores
ax1 = axes[0,0]
top12 = df_forecast.head(12).copy()
bcolors = top12['forecast_status'].map(SC).fillna('#90A4AE')
bars = ax1.barh(top12['item_name'][::-1], top12['urgency_score'][::-1], color=bcolors[::-1], edgecolor='white', height=.7)
ax1.axvline(x=70, color='#E53935', linestyle='--', alpha=.5, linewidth=1)
ax1.axvline(x=50, color='#FB8C00', linestyle='--', alpha=.5, linewidth=1)
for bar in bars:
    w = bar.get_width()
    ax1.text(w+1, bar.get_y()+bar.get_height()/2, f'{w:.0f}', va='center', fontsize=9)
ax1.set_xlabel('Urgency Score (0–100)'); ax1.set_title('Top 12 by Urgency Score', fontweight='bold')
ax1.legend(handles=[mpatches.Patch(color=c,label=s) for s,c in SC.items()], fontsize=8, loc='lower right')
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# Chart 2: Scatter urgency vs days
ax2 = axes[0,1]
for status, color in SC.items():
    mask = df_forecast['forecast_status']==status
    ax2.scatter(df_forecast.loc[mask,'urgency_score'], df_forecast.loc[mask,'forecast_days_to_empty'],
                c=color, label=status, alpha=.8, s=70, edgecolors='white', linewidths=.5)
for _, row in df_forecast.head(5).iterrows():
    ax2.annotate(row['item_name'], (row['urgency_score'],row['forecast_days_to_empty']), fontsize=7, xytext=(5,3), textcoords='offset points')
ax2.axhline(y=7, color='#FB8C00', linestyle='--', alpha=.5, linewidth=1)
ax2.axhline(y=3, color='#E53935', linestyle='--', alpha=.5, linewidth=1)
ax2.set_xlabel('Urgency Score'); ax2.set_ylabel('Days to Empty')
ax2.set_title('Urgency vs Days to Empty', fontweight='bold'); ax2.legend(fontsize=8)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

# Chart 3: Spend by category
ax3 = axes[1,0]
cats = list(spend['by_category'].keys()); vals = list(spend['by_category'].values())
pcts = list(spend['by_category_pct'].values())
palette = plt.cm.Set2(np.linspace(0,1,len(cats)))
bars3 = ax3.barh(cats[::-1], vals[::-1], color=palette[::-1], edgecolor='white', height=.7)
for bar, pct in zip(bars3, pcts[::-1]):
    w = bar.get_width()
    ax3.text(w+20, bar.get_y()+bar.get_height()/2, f'₹{int(w):,} ({pct:.0f}%)', va='center', fontsize=8)
ax3.set_title(f'Monthly Spend by Category | Total ₹{spend["total_monthly_est"]:,.0f}', fontweight='bold')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

# Chart 4: Menu coverage
ax4 = axes[1,1]
cov20 = df_coverage.head(20).copy()
rmap  = {'HIGH RISK':'#E53935','MEDIUM RISK':'#FB8C00','LOW RISK':'#FDD835','COVERED':'#43A047'}
ccolors = cov20['risk_level'].map(rmap).fillna('#90A4AE')
ax4.barh(cov20['item_name'][::-1], cov20['coverage_pct'][::-1], color=ccolors[::-1], edgecolor='white', height=.7)
ax4.axvline(x=100, color='#43A047', linestyle='--', alpha=.6, linewidth=1.5)
ax4.set_xlabel('Coverage %'); ax4.set_title('Menu Coverage — 20 Most Constrained', fontweight='bold')
ax4.legend(handles=[mpatches.Patch(color=c,label=l) for l,c in rmap.items()], fontsize=8)
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sprint2_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Dashboard saved → {OUTPUT_DIR}/sprint2_dashboard.png')

## Cell 8 — Save to GitHub
File → Save a copy in GitHub, OR run this cell to commit from Colab.

In [ ]:
import subprocess
from datetime import datetime

def push_to_github(commit_message=None):
    if not commit_message:
        commit_message = f'Sprint 2 outputs: {datetime.now().strftime("%Y-%m-%d %H:%M")}'
    cmds = ['git add -A', f'git commit -m "{commit_message}"', 'git push']
    for cmd in cmds:
        result = subprocess.run(cmd, shell=True, cwd=REPO_DIR, capture_output=True, text=True)
        status = '✓' if result.returncode==0 or 'nothing to commit' in result.stdout else '⚠️'
        print(f'{status} {cmd}')
        if result.returncode != 0 and 'nothing to commit' not in result.stdout:
            print(f'   {result.stderr.strip()}')

# ── Option A: Use Colab's built-in GitHub save (recommended — no token needed)
# File → Save a copy in GitHub → select your repo → commit message → OK
print('RECOMMENDED: File → Save a copy in GitHub')
print()

# ── Option B: push via git (needs GITHUB_TOKEN in Colab Secrets)
# Uncomment below if you prefer terminal-style push
# push_to_github('Sprint 2: Analytics engine complete')

print('=' * 60)
print('  SmartPantry AI — Sprint 2 COMPLETE ✓')
print('=' * 60)
print()
print(f'  Forecast : {len(df_forecast)} items scored')
print(f'  Orders   : {len(df_order)} items | ₹{total_cost:,.0f}')
print(f'  Coverage : {covered}/{len(df_coverage)} menu ingredients covered')
print(f'  Spend    : ₹{spend["total_monthly_est"]:,.0f}/mo ({spend["utilisation_pct"]}% of budget)')
print()
print('  Ready for Sprint 3: Claude API Agent Brain')
print(f'  Repo: https://github.com/{GITHUB_USER}/{GITHUB_REPO}')